<a href="https://colab.research.google.com/github/bhaviii123/Air_Passenger_Forecasting_ML_vs_DL.ipynb/blob/main/Copy_of_architecture_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. model.py (The Architecture)

This file contains the Channel Attention and the Convolutional Layers ($32, 64, 128, 256, 512$). It uses ReLU for hidden layers and Sigmoid for the attention gate.

In [ ]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False),
            nn.Sigmoid() # Produces weights from 0 to 1
        )

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.fc(y)
        return x * y # Multiplies weights by original patch

class DenoiseNet(nn.Module):
    def __init__(self):
        super(DenoiseNet, self).__init__()
        # Input RGB (3 channels) -> 64 Filters
        self.layer1 = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1), nn.ReLU())

        # Increasing Channels: 64 -> 128 -> 256 -> 512
        self.layer2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU())
        self.layer3 = nn.Sequential(nn.Conv2d(128, 256, 3, padding=1), nn.ReLU())
        self.attention = ChannelAttention(256) # Attention on 256-channel map

        self.layer4 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.ReLU())
        self.layer5 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.ReLU())

        # Output layer back to RGB
        self.output = nn.Conv2d(64, 3, 3, padding=1)

    def forward(self, x):
        residual = x
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.attention(out) # Focus on important parts
        out = self.layer4(out)
        out = self.layer5(out)
        out = self.output(out)
        return out + residual # Clean = Noisy Input + Learned Residual

2. dataset.py (Data & Noise Handling)

This handles the BSDS500 images, creates 256x256 patches, and adds the Sigma ($\sigma$) noise.

In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import numpy as np

def get_dataloader(data_path, batch_size=16, patch_size=256):
    transform = transforms.Compose([
        transforms.RandomCrop(patch_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), # Normalizes to [0, 1]
    ])

    dataset = datasets.ImageFolder(root=data_path, transform=transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

def add_noise(img, sigma):
    # sigma is 10, 20, 30, 40, or 50
    noise = torch.randn(img.size()) * (sigma / 255.0)
    return torch.clamp(img + noise, 0, 1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to where your BSDS500 images actually are on your Drive
TRAIN_PATH = '/content/drive/MyDrive/data/bsr/BSDS500/image/train'

Mounted at /content/drive


In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class BSDS500Dataset(Dataset):
    def __init__(self, root_dir, patch_size=256):
        self.root_dir = root_dir
        # Only list files that are images
        self.image_filenames = [f for f in os.listdir(root_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        self.transform = transforms.Compose([
            transforms.RandomCrop(patch_size),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(), # Converts to [0, 1] range
        ])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        return image, 0 # Return 0 as a dummy label to keep training loop happy

def get_dataloader(data_path, batch_size=16, patch_size=256):
    dataset = BSDS500Dataset(root_dir=data_path, patch_size=patch_size)
    if len(dataset) == 0:
        raise FileNotFoundError(f"No images found in {data_path}. Check your path!")
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

3. train.py (The Execution Script)

This is your main script. It runs for 500 epochs and saves the model to your path.

In [ ]:
test_sigmas = [10, 20, 30, 40, 50]
for s in test_sigmas:
    # Add noise, run model, calculate PSNR
    print(f"Testing Sigma: {s}...")

Testing Sigma: 10...
Testing Sigma: 20...
Testing Sigma: 30...
Testing Sigma: 40...
Testing Sigma: 50...
